In [1]:
!pip install -q torch transformers datasets tqdm

In [2]:
import torch
from transformers import LongformerForQuestionAnswering, LongformerTokenizerFast
from datasets import load_dataset
from torch.utils.data import DataLoader
from torch.optim import AdamW
from tqdm import tqdm

In [3]:
model_name = "allenai/longformer-base-4096"

tokenizer = LongformerTokenizerFast.from_pretrained(model_name)
model = LongformerForQuestionAnswering.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/597M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/269 [00:00<?, ?it/s]

LongformerForQuestionAnswering LOAD REPORT from: allenai/longformer-base-4096
Key                            | Status     | 
-------------------------------+------------+-
lm_head.decoder.weight         | UNEXPECTED | 
lm_head.bias                   | UNEXPECTED | 
longformer.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.weight           | UNEXPECTED | 
lm_head.dense.bias             | UNEXPECTED | 
lm_head.layer_norm.bias        | UNEXPECTED | 
longformer.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.weight      | UNEXPECTED | 
qa_outputs.weight              | MISSING    | 
qa_outputs.bias                | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [4]:
# Tải SQuAD v1.1
dataset = load_dataset("squad")

# Xem tổng quan cấu trúc
print(dataset)

# Xem mẫu đầu tiên
print(dataset["train"][0])

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/1.82M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 87599
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 10570
    })
})
{'id': '5733be284776f41900661182', 'title': 'University_of_Notre_Dame', 'context': 'Architecturally, the school has a Catholic character. Atop the Main Building\'s gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome

In [5]:
def preprocess_function(rows):
    questions = [q.strip() for q in rows["question"]]
    contexts = rows["context"]
    answers = rows["answers"]

    # ═══ BƯỚC 6.1: Tokenize question + context ═══
    # truncation="only_second" → chỉ cắt phần context nếu quá dài
    # return_offsets_mapping=True → nhận offset (start, end) của từng token
    encodings = tokenizer(
        questions,
        contexts,
        max_length=4096,
        truncation="only_second",
        padding="max_length",
        return_offsets_mapping=True
    )

    # encodings trả về: {input_ids, attention_mask, offset_mapping, ...}

    start_position = []
    end_position = []

    # ═══ BƯỚC 6.2: Chuyển answer từ character → token position ═══
    for i, offset_mapping in enumerate(encodings["offset_mapping"]):
        answer = answers[i]
        start_char = answer["answer_start"][0]
        end_char = start_char + len(answer["text"][0])

        token_start_index = 0
        token_end_index = 0

        # Duyệt từng (start, end) trong offset_mapping
        for idx, (start, end) in enumerate(offset_mapping):
            if start <= start_char < end:
                token_start_index = idx    # token bao trùm start_char
            if start < end_char <= end:
                token_end_index = idx      # token bao trùm end_char
                break

        start_position.append(token_start_index)
        end_position.append(token_end_index)

    # ═══ BƯỚC 6.3: Gắn nhãn vào encodings ═══
    encodings.update({
        "start_position": start_position,
        "end_position": end_position
    })

    return encodings

In [6]:
# Áp dụng hàm preprocessing lên train + validation
encoded_dataset = dataset.map(preprocess_function, batched=True)

# Chuyển các cột cần thiết sang dạng PyTorch tensor
encoded_dataset.set_format("torch", columns=[
    "input_ids",
    "attention_mask",
    "start_position",
    "end_position"
])

Map:   0%|          | 0/87599 [00:00<?, ? examples/s]

Map:   0%|          | 0/10570 [00:00<?, ? examples/s]

In [14]:
train_dataset = encoded_dataset["train"]
val_dataset = encoded_dataset["validation"]

# Chỉ lấy subset để chạy nhanh (demo)
train_dataloader = DataLoader(
    train_dataset.select(range(2000)),   # 2000 mẫu train
    batch_size=4,
    shuffle=True
)
val_dataloader = DataLoader(
    val_dataset.select(range(500)),     # 500 mẫu validation
    batch_size=4
)

In [8]:
# Auto-detect: dùng GPU nếu có, CPU nếu không
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
model.to(device)
print(device)  # → cuda hoặc cpu

# AdamW optimizer với learning rate nhỏ (fine-tuning)
optimizer = AdamW(model.parameters(), lr=2e-5)

cuda


In [15]:
num_epochs = 2

for epoch in range(num_epochs):
    model.train()   # Bật chế độ train (kích hoạt dropout, BN...)
    print(f"Epoch {epoch + 1}/{num_epochs}")

    total_loss = 0
    for batch in tqdm(train_dataloader):

        # 1. Lấy dữ liệu và chuyển sang device
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        start_position = batch["start_position"].to(device)
        end_position = batch["end_position"].to(device)

        # 2. Xóa gradient từ batch trước
        optimizer.zero_grad()

        # 3. Forward pass — mô hình tự tính loss khi có start/end positions
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            start_positions=start_position,
            end_positions=end_position
        )

        loss = outputs.loss        # Cross-entropy loss cho start + end
        total_loss += loss.item()

        # 4. Backward pass — tính gradient
        loss.backward()

        # 5. Cập nhật trọng số
        optimizer.step()

    average_loss = total_loss / len(train_dataloader)
    print(f"Train loss: {average_loss:.4f}")

Epoch 1/2


100%|██████████| 500/500 [10:46<00:00,  1.29s/it]


Train loss: 0.6250
Epoch 2/2


100%|██████████| 500/500 [10:46<00:00,  1.29s/it]

Train loss: 0.4270


In [16]:
model.eval()    # Tắt dropout, BN về eval mode
correct = 0
total = 0

with torch.no_grad():    # Không tính gradient — tiết kiệm memory
    for batch in tqdm(val_dataloader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        start_position = batch["start_position"].to(device)
        end_position = batch["end_position"].to(device)

        # Forward KHÔNG có start/end_positions → chỉ trả logits
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # Lấy vị trí có score cao nhất
        predicted_start = torch.argmax(outputs.start_logits, dim=1)
        predicted_end = torch.argmax(outputs.end_logits, dim=1)

        # Đúng khi CẢ HAI start và end khớp chính xác
        correct += ((predicted_start == start_position) &
                    (predicted_end == end_position)).sum().item()
        total += start_position.size(0)

accuracy = correct / total
print(f"Validation accuracy: {accuracy:.4f}")

100%|██████████| 125/125 [00:47<00:00,  2.63it/s]

Validation accuracy: 0.5520


In [17]:
model.to(device)
model.eval()   # Quan trọng: phải ở eval mode trước inference

LongformerForQuestionAnswering(
  (longformer): LongformerModel(
    (embeddings): LongformerEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(4098, 768, padding_idx=1)
    )
    (encoder): LongformerEncoder(
      (layer): ModuleList(
        (0-11): 12 x LongformerLayer(
          (attention): LongformerAttention(
            (self): LongformerSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (query_global): Linear(in_features=768, out_features=768, bias=True)
              (key_global): Linear(in_features=768, out_features=768, bias=True)
              (

In [18]:
def predict_answer(question, context):
    # Bước 1: Tokenize question + context
    inputs = tokenizer(
        question,
        context,
        max_length=4096,
        truncation=True,
        padding="max_length",
        return_tensors="pt"   # Trả về PyTorch tensor
    )

    # Bước 2: Chuyển sang device
    input_ids = inputs["input_ids"].to(device)
    attention_mask = inputs["attention_mask"].to(device)

    # Bước 3: Inference không gradient
    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)

        # Bước 4: Lấy vị trí có điểm cao nhất
        start_index = torch.argmax(outputs.start_logits, dim=1).item()
        end_index = torch.argmax(outputs.end_logits, dim=1).item()

        # Bước 5: Cắt span token và decode về text
        answer_tokens = input_ids[0, start_index:end_index + 1]
        answer = tokenizer.decode(answer_tokens, skip_special_tokens=True)

    return answer, start_index, end_index

In [19]:
question = "Who is the president of the United States in 2023?"
context = (
    "In 2023, the president of the United States was Joe Biden. "
    "He took office on January 20, 2021, following his victory "
    "in the 2020 presidential election. His administration focused on "
    "climate change, healthcare reform, and economic recovery "
    "following the COVID-19 pandemic."
)

answer, start_index, end_index = predict_answer(question, context)

print(f"Question:    {question}")
print(f"Answer:      {answer}")
print(f"Start index: {start_index}")
print(f"End index:   {end_index}")

Question:    Who is the president of the United States in 2023?
Answer:       Joe Biden
Start index: 26
End index:   27


In [20]:
question = "When did Biden take office?"
context = (
    "In 2023, the president of the United States was Joe Biden. "
    "He took office on January 20, 2021, following his victory "
    "in the 2020 presidential election. His administration focused on "
    "climate change, healthcare reform, and economic recovery "
    "following the COVID-19 pandemic."
)

answer, start_index, end_index = predict_answer(question, context)

print(f"Question:    {question}")
print(f"Answer:      {answer}")
print(f"Start index: {start_index}")
print(f"End index:   {end_index}")

Question:    When did Biden take office?
Answer:       January 20, 2021
Start index: 27
End index:   30


In [21]:
question = "What was Biden's administration focused on?"
context = (
    "In 2023, the president of the United States was Joe Biden. "
    "He took office on January 20, 2021, following his victory "
    "in the 2020 presidential election. His administration focused on "
    "climate change, healthcare reform, and economic recovery "
    "following the COVID-19 pandemic."
)

answer, start_index, end_index = predict_answer(question, context)

print(f"Question:    {question}")
print(f"Answer:      {answer}")
print(f"Start index: {start_index}")
print(f"End index:   {end_index}")

Question:    What was Biden's administration focused on?
Answer:       climate change, healthcare reform, and economic recovery
Start index: 47
End index:   55
